# 06 · Demo bundle (scored, hashed, ready for the laptop)

**Accelerator:** none (CPU scoring mirrors the laptop) · **Runtime:** ~15 min · **Attach:** `abo-prepared`, `lv-features-checkpoints`, `lv-consistency`, the `abo-fakes-*` datasets, optional `phone-set` · **Output:** `demo_bundle.zip`

Copies the heads, thresholds and BOTH calibrators, picks a structured example set (reals with claims, one t2i and
one i2i fake per generator, a count>1 item, phone items), scores every example at JPEG none/70/50/30 in the
recommended mode with and without the VLM, and records expected outputs, latency and SHA-256 hashes.
Then fill `visible_ok` in `examples.csv` by eye. On the laptop: `docs/DEMO.md` (one online prefetch of CLIP and
SmolVLM, then three offline rehearsals from a fresh process).

## 0. Setup (same cell in every notebook)

**Before running:** attach the code as a Kaggle Dataset named `listing-verifier-code`
(upload the repo folder without `data/`, `outputs/`, `web/node_modules/`), **or** set `REPO_URL`
to your GitHub repo. Turn on **Internet** (pip, ABO and HF downloads). For the GPU notebooks pick
**Settings → Accelerator → GPU T4 x2**. Store your Hugging Face token as the Kaggle secret `HF_TOKEN`.

In [ ]:
import os, shutil, subprocess, sys, pathlib, json
REPO_URL = "https://github.com/kvamsi-iisc/DLVA-Final-Project-Forensic-Verification-of-Marketplace-Listings.git"   # used if the code dataset is not attached
WORK = pathlib.Path("/kaggle/working"); REPO = WORK / "repo"; DATA = WORK / "data"; OUT = WORK / "outputs"
code_ds = pathlib.Path("/kaggle/input/listing-verifier-code")
if REPO.exists() and (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "-q"], check=False)     # pick up code fixes in a reused session
elif not REPO.exists():
    if code_ds.exists():
        shutil.copytree(code_ds, REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
os.environ["LV_DATA_ROOT"] = str(DATA); os.environ["LV_OUTPUT_ROOT"] = str(OUT)
os.environ["HF_HOME"] = str(WORK / "hf_cache")           # model weights land here (removed before publishing)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret:", type(e).__name__, "(fine unless a gated model is needed)")
print("python", sys.version.split()[0], "| code:", subprocess.run(["git", "-C", str(REPO), "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())
pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], capture_output=True, text=True)
if pip.returncode != 0:
    print(pip.stdout[-3000:], pip.stderr[-3000:])          # show WHY before stopping (e.g. a pin without a wheel for this Python)
    raise SystemExit("pip install failed; see output above")
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)
sys.path.insert(0, str(REPO))
from listing_verifier.kaggle import init, run, attach, optional_attach, merge_fakes   # helpers live in the repo: fixes need no re-import
WORK, REPO, DATA, OUT, LOGS = init(WORK)
run("scripts/env_report.py", "--out", OUT / "results" / "env_report.json")
import torch; print(f"visible GPUs: {torch.cuda.device_count()}")
print("attached inputs:", sorted(p.name for p in pathlib.Path("/kaggle/input").iterdir()) if pathlib.Path("/kaggle/input").exists() else [])


In [ ]:
attach("abo-prepared", "data/processed", DATA / "processed", must_contain="abo_clean.csv")
attach("lv-features-checkpoints", "outputs/checkpoints", OUT / "checkpoints", must_contain="main")
cal = pathlib.Path("/kaggle/input/lv-consistency/outputs/checkpoints/main")
if cal.exists():   # ONLY the consistency-stage artifacts; never let an older copy of the heads overwrite the authenticity models
    for f in cal.glob("calibrator*"):
        shutil.copy(f, OUT / "checkpoints" / "main" / f.name); print("calibrator:", f.name)
FAKE_SLUGS = ['abo-fakes-pilot', 'abo-fakes-turbo-t2i', 'abo-fakes-turbo-i2i', 'abo-fakes-sd15-i2i', 'abo-fakes-kand-t2i', 'abo-fakes-kand-i2i']
merge_fakes(FAKE_SLUGS, DATA / "fakes")
optional_attach("phone-set", "phone", DATA / "phone", must_contain="labels.csv")
run("scripts/05_build_index.py", "--no-audit")
run("scripts/12_build_demo_bundle.py", "--tag", "main", "--n-examples", 12)
print(json.dumps(json.load(open(OUT / "demo_bundle" / "manifest.json")), indent=1))
shutil.make_archive(str(WORK / "demo_bundle"), "zip", OUT / "demo_bundle"); print("zip ready: /kaggle/working/demo_bundle.zip")

## Persisting outputs
When this notebook finishes under **Save Version → Save & Run All (Commit)**, everything in `/kaggle/working`
except the model cache is kept as the notebook output. Then: *Output tab → "New Dataset"* and name it as stated
at the top of this notebook. The next notebook attaches that dataset. The cell below checks the artifacts this
notebook promised actually exist, then removes the model cache so the output stays small.

In [ ]:
required = ["demo_bundle.zip", "outputs/demo_bundle/examples/expected.json"]
run("scripts/verify_outputs.py", *[WORK / r for r in required])   # exits non-zero on a missing/empty artifact
shutil.rmtree(WORK / "hf_cache", ignore_errors=True)
shutil.rmtree(REPO / "web" / "node_modules", ignore_errors=True)
print(subprocess.run(["du", "-sh", str(DATA), str(OUT)], capture_output=True, text=True).stdout)